## Overall workflow
1. Environment and library setup
2. Load and inspect the dataset
3. Data cleaning
4. Feature engineering
5. Feature/target separation
6. Preprocessing pipelines
7. Train/test split
8. Baseline model comparison
9. Final Gradient Boosting pipeline
10. Hyperparameter tuning and cross-validation
11. Final evaluation and model persistence
12. Feature importance
13. Visualization assets
14. Real-data visualization from the saved model


# 1. Imports

**Why this step is used:** Load the libraries required by the ML workflow. Imports are kept in one place so later cells can directly use the same objects.

**Steps:**
- Data handling: pandas, numpy
- Visualization: matplotlib, seaborn
- Preprocessing and modeling: scikit-learn
- Model persistence: joblib


In [ ]:
import os
# File and directory handling

import matplotlib.pyplot as plt
# Create static data visualizations and plots (line charts, scatter plots, histograms)

import numpy as np
# Perform high-performance numerical operations and multi-dimensional array processing

import pandas as pd
# Load, clean, manipulate, and analyze tabular data (DataFrames and Series)

import seaborn as sns
# Create advanced statistical visualizations built on top of Matplotlib (heatmaps, distributions)

from sklearn.compose import ColumnTransformer
# Apply different feature preprocessing transformers to specific columns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
# Import tree-based ensemble regression models for predictive modeling

from sklearn.impute import SimpleImputer
# Handle missing or NaN values

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Calculate model performance evaluation metrics

from sklearn.model_selection import RandomizedSearchCV, train_test_split
# Split data and perform randomized hyperparameter tuning

from sklearn.pipeline import Pipeline
# Chain preprocessing steps and model training into a single executable object

from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Encode categorical features and standardize numerical features

import joblib
# Save and load trained machine-learning models


# 2. Load Dataset

**Why this step is used:** Load the historical food-delivery dataset that provides the input features and target variable for model training.


In [2]:
df = pd.read_csv("Food_Delivery_Time_Prediction.csv")

# 3. Data Inspection / EDA

**Why this step is used:** Understand the dataset before modeling by checking its size, missing values, data types, and target distribution.

**Steps:**
- Inspect dataset shape
- Check missing values
- Check data types
- Visualize Time_taken_min distribution


In [3]:
print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Dataset Shape: (50000, 24)

Missing Values:
 Order_ID                      0
Order_Date                    0
Order_Hour                    0
Day_of_Week                   0
Is_Weekend                    0
Is_Festival                   0
Weather                       0
Pickup_Zone                   0
Dropoff_Zone                  0
Vehicle_Type                  0
Rider_Experience_Years        0
Rider_Rating                  0
Restaurant_Rating             0
Cuisine_Type                  0
Order_Items                   0
Restaurant_Load               0
Preparation_Time_Min          0
Road_Distance_km              0
Delivery_Distance_Category    0
Traffic_Level                 0
Number_of_Signals             0
Average_Speed_kmph            0
Delivery_Priority             0
Time_taken_min                0
dtype: int64

Data Types:
 Order_ID                          str
Order_Date                        str
Order_Hour                      int64
Day_of_Week                       str
Is_Weeke

In [4]:
# Visualizing target distribution
sns.histplot(df["Time_taken_min"], kde=True)
plt.title("Target Distribution: Time Taken (min)")
plt.savefig("target_distribution.png")
plt.close()

# 4. Data Cleaning

**Why this step is used:** Remove fields that are directly identified in the supplied code as not required for modeling: the unique order identifier and the date string.


In [5]:
df_clean = df.drop(columns=["Order_ID", "Order_Date"])

# 5. Feature Engineering

**Why this step is used:** Create a domain-based travel-time feature from road distance and average speed, then separate the predictors from the target and identify numerical and categorical columns.

**Steps:**
- Create Estimated_Travel_Time
- Separate X and y
- Identify numerical columns
- Identify categorical columns


In [6]:
df_clean["Estimated_Travel_Time"] = (
    df_clean["Road_Distance_km"] / np.maximum(df_clean["Average_Speed_kmph"], 1)
) * 60

X = df_clean.drop(columns=["Time_taken_min"])
y = df_clean["Time_taken_min"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

/var/folders/8b/wmmwfhsx0331797dlyym7vdc0000gn/T/ipykernel_96012/3566171095.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


# 6. Preprocessing Pipeline

**Why this step is used:** Prepare numerical and categorical features consistently before they enter the regression model.

**Steps:**
- Numerical features: median imputation + StandardScaler
- Categorical features: most-frequent imputation + OneHotEncoder
- ColumnTransformer applies each pipeline to the correct columns


In [7]:
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ]
)

# 7. Train-Test Split

**Why this step is used:** Separate the dataset into training and testing portions so model performance can be evaluated on data not used during training.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. Baseline Models

**Why this step is used:** Compare two tree-based regression approaches using the same preprocessing pipeline before moving to the selected Gradient Boosting configuration.

**Steps:**
- Random Forest Regressor
- Gradient Boosting Regressor
- Evaluate using RMSE, MAE, and R²


In [9]:
baseline_models = {
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

print("\n--- Baseline Model Evaluation ---")
for name, model in baseline_models.items():
    full_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("regressor", model)]
    )

    full_pipeline.fit(X_train, y_train)
    preds = full_pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"{name} -> RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}")


--- Baseline Model Evaluation ---
Random Forest -> RMSE: 3.040 | MAE: 2.374 | R2: 0.993
Gradient Boosting -> RMSE: 2.883 | MAE: 2.294 | R2: 0.993


# 9. Gradient Boosting Model

**Why this step is used:** Build the main Gradient Boosting pipeline with preprocessing and the supplied model parameters, then train it on the training data.

**Steps:**
- Reuse the existing training columns
- Build the preprocessing transformer
- Create the GradientBoostingRegressor
- Fit the complete pipeline


In [10]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_cols,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            GradientBoostingRegressor(
                n_estimators=150,
                learning_rate=0.08,
                max_depth=4,
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

/var/folders/8b/wmmwfhsx0331797dlyym7vdc0000gn/T/ipykernel_96012/3505431606.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['Order_Hour','Day_of_Week','Is_Weekend',...,'Average_Speed_kmph', 'Delivery_Priority','Estimated_Travel_Time']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automati

# 10. Hyperparameter Tuning

**Why this step is used:** Search across the supplied Gradient Boosting hyperparameter ranges using RandomizedSearchCV and 5-fold cross-validation to identify a stronger model configuration.

**Steps:**
- Define tuning pipeline
- Define parameter distributions
- Run RandomizedSearchCV
- Select best_model and best parameters


In [11]:
tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", GradientBoostingRegressor(random_state=42)),
    ]
)

param_dist = {
    "regressor__n_estimators": [100, 200, 300],
    "regressor__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "regressor__max_depth": [3, 5, 7],
    "regressor__min_samples_split": [2, 5, 10],
}

search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("\n--- Hyperparameter Tuning Complete ---")
print("Best Parameters:", search.best_params_)


--- Hyperparameter Tuning Complete ---
Best Parameters: {'regressor__n_estimators': 200, 'regressor__min_samples_split': 5, 'regressor__max_depth': 5, 'regressor__learning_rate': 0.1}


# 11. Final Evaluation

**Why this step is used:** Evaluate the tuned best_model on the held-out test set using the same regression performance measures used in the supplied workflow.


In [12]:
final_preds = best_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))
final_r2 = r2_score(y_test, final_preds)

print(
    f"Final Model Performance -> Test RMSE: {final_rmse:.3f} | Test R2: {final_r2:.3f}"
)

Final Model Performance -> Test RMSE: 2.711 | Test R2: 0.994


# 12. Feature Importance

**Why this step is used:** Inspect which transformed features contribute most to the Gradient Boosting model represented by the existing model variable.


In [13]:
importances = pd.Series(
    model.named_steps["regressor"].feature_importances_,
    index=preprocessor.get_feature_names_out(),
).sort_values(ascending=False)

print(importances.head(10))

num__Estimated_Travel_Time       0.938014
num__Preparation_Time_Min        0.055485
num__Number_of_Signals           0.002669
num__Average_Speed_kmph          0.001539
cat__Weather_Storm               0.000715
cat__Weather_Rain                0.000588
num__Is_Festival                 0.000459
cat__Traffic_Level_Low           0.000189
cat__Delivery_Priority_Normal    0.000130
num__Road_Distance_km            0.000092
dtype: float64


# 13. Visualization

**Why this step is used:** Generate visual assets from the actual CSV data and the trained pipeline. The separate synthetic demonstration-data block was removed because it overwrote the main `df` and `model` variables and was not part of the actual training workflow.

**Steps:**
- Correlation heatmap from the real dataset
- Top-10 feature importance chart from the trained pipeline
- Save charts under the assets directory


In [16]:
# =========================================================
# Actual vs Predicted Delivery Times
# =========================================================

os.makedirs("assets", exist_ok=True)

plt.style.use("dark_background")

# Generate predictions using the final tuned model
y_pred = best_model.predict(X_test)

# Create Actual vs Predicted plot
plt.figure(figsize=(10, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.6,
    color="#38bdf8",
    edgecolors="none",
    s=45,
    label="Predictions",
)

# Ideal prediction line (Actual = Predicted)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="#f43f5e",
    linestyle="--",
    linewidth=2.5,
    label="Ideal Fit",
)

# Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Title
plt.title(
    "Actual vs. Predicted Delivery Times",
    fontsize=18,
    pad=15,
    fontweight="bold",
)

# Axis labels
plt.xlabel(
    "Actual Time (min)",
    fontsize=14,
)

plt.ylabel(
    "Predicted Time (min)",
    fontsize=14,
)

# Add metrics to the graph
plt.text(
    0.05,
    0.95,
    f"RMSE: {rmse:.2f}\nMAE: {mae:.2f}\nR²: {r2:.3f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="black",
        alpha=0.65,
        edgecolor="white",
    ),
)

plt.legend(
    loc="upper left",
    fontsize=11,
)

plt.tight_layout()

# Save graph
plt.savefig(
    "assets/actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()

print("Successfully generated Actual vs Predicted chart!")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R²: {r2:.3f}")

Successfully generated Actual vs Predicted chart!
RMSE: 2.711
MAE: 2.141
R²: 0.994


# 14. Save Model

**Why this step is used:** Persist the tuned best model so the trained pipeline can be loaded later by the application/backend without retraining.


In [15]:
joblib.dump(best_model, "delivery_time_model.pkl")
print("Model saved to delivery_time_model.pkl")

Model saved to delivery_time_model.pkl
